# TD-DFTB-based NBRA-workflow for DFTB+

In this tutorial, we demonstrate some some higher-level functions to streamline DFTB+ calculations of the properties needed for NA-MD simulations. We will use these functions to define a workflow for NBRA calculations with DFTB+ at the TD-DFTB level of electronic structure. While the resulting function is a prototype for the non-NBRA calculations, at this point we will only focus on its use for NBRA calculations.

## Table of contents
<a name="toc"></a>
1. [Writing Gen-files](#1)
2. [Writing input file for DFTB+, `dftb_in.hsd`](#2)
3. [Needed Slater-Koster files and system-specific parameters](#3)
4. [Run DFTB+ calculations](#4)

   4.1. [To generate H and S](#4.1)
   
   4.2. [To do SCF and TD-DFT calculations](#4.2)
   
   4.3. [The overlap for the doubled-molecule - ODIN](#4.3)
   
   4.4. [Check time-overlap](#4.4)
   
   4.5. [Extracting MO/CI data in the required format](#4.5)
   
5. [Putting everything together](#5)
    
   5.1. [Develop the function](#5.1)
   
   5.2. [Test the funciton](#5.2)
   
   5.3. [Test Libra implementation](#5.3)
   

### A. Learning objectives

* To create gen files for DFTB+ calculations
* To create DFTB+ input files 
* To run ODIN calculations of atomic overlaps
* To compute time-overlaps of AOs using ODIN
* To extract the key infromation from the TD-DFTB calculations from DFTB+ output (energies, MOs, etc.)
* To compute NACs with TD-DFTB excited states
* To execute NBRA workflow with DFTB+


### B. Use cases

* process the DFTB+ calculations results
* compute wavefunction time-overlaps with DFTB+
* NBRA workflow with DFTB+


### C. Functions

- `libra_py`
  - `packages`
    - `dftbplus`
      - `methods`
        - [`create_odin_inp`](#create_odin_inp-1)
        - [`dftb_compute_adi`](#dftb_compute_adi-1)
        - [`make_dftb_input`](#make_dftb_input-1)
        - [`read_dftb_orbital_info`](#read_dftb_orbital_info-1)
        - [`read_overlap_matrix`](#read_overlap_matrix-1)
        - [`read_spx_mappings`](#read_spx_mappings-1)
        - [`run_dftb`](#run_dftb-1)
        - [`run_odin`](#run_odin-1)
        - [`write_dftb_gen`](#write_dftb_gen-1)


In [1]:
import os
import sys
import re
import copy
import numpy as np
import scipy
from liblibra_core import MATRIX, CMATRIX, CMATRIXList, Py2Cpp_int, Cpp2Py

# Fisrt, we add the location of the library to test to the PYTHON path
import libra_py.packages.dftbplus.methods as DFTB_methods
import libra_py.packages.cp2k. methods as cp2k
import libra_py.citools.ci as ci
from libra_py import data_conv
from libra_py import units
import libra_py.orthogonalizations as ortho

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

## 1. Writing Gen-files
<a name="1"></a>[Back to TOC](#toc)

<a name="write_dftb_gen-1"></a>

In [2]:
help(DFTB_methods.write_dftb_gen)

Help on function write_dftb_gen in module libra_py.packages.dftbplus.methods:

write_dftb_gen(filename, atom_labels, coordinates, periodic=False)
    Write a DFTB+ .gen file.
    
    Parameters
    ----------
    filename : str
        Output file name.
    atom_labels : list of str
        Atomic symbols, length N.
    coordinates : array-like, shape (3N, 1)
        Flattened Cartesian coordinates:
        [x1,y1,z1,x2,y2,z2,...]^T  (Angstrom)
    periodic : bool
        True -> periodic system (S)
        False -> cluster (C)
    
    Example
    -------
    > atom_labels = ["O", "H", "H"]
    > 
    > coordinates = np.array([
    > 0.000000, 0.000000, 0.000000,
    > 0.758602, 0.000000, 0.504284,
    > -0.758602, 0.000000, 0.504284
    > ]).reshape(-1, 1)
    >
    > write_dftb_gen("water.gen", atom_labels, coordinates)



In [3]:
atom_labels = ["O", "H", "H"]

coordinates = np.array([
    0.000000, 0.000000, 0.000000,
    0.758602, 0.000000, 0.504284,
   -0.758602, 0.000000, 0.504284
]).reshape(-1, 1)

DFTB_methods.write_dftb_gen("water.gen", atom_labels, coordinates)

DFTB+ GEN file written to water.gen


## 2. Writing input file for DFTB+, `dftb_in.hsd`
<a name="2"></a>[Back to TOC](#toc)

<a name="make_dftb_input-1"></a>

In [4]:
help(DFTB_methods.make_dftb_input)

Help on function make_dftb_input in module libra_py.packages.dftbplus.methods:

make_dftb_input(params, ground_state=True, working_directory='.')
    Generate a DFTB+ input file ("dftb_in.hsd") in a thread-safe manner.
    
    This function constructs a complete DFTB+ input file for either
    ground-state SCC-DFTB or excited-state TD-DFTB (Casida formalism)
    calculations and writes it to a specified working directory.
    
    Unlike earlier implementations, this version avoids reliance on the
    global working directory and is safe for parallel execution (e.g.,
    multi-trajectory dynamics), where each calculation runs in its own
    directory.
    
    The generated input file includes the following sections:
    
        - Geometry (GenFormat)
        - Driver
        - Hamiltonian (SCC-DFTB)
        - ExcitedState (Casida TD-DFTB, if requested)
        - Options
        - Analysis
    
    Parameters
    ----------
    params : dict
        Dictionary containing DFTB+ input 

In [5]:
dftb_params = {
    "gen_file" : "x1.gen",
    "sk_prefix" : "../mio/FinalSK/",
    "Driver" : "{}",
    "MaxAngularMomentum" : """{ O = "p" 
                          H = "s" 
                     }
                     """,
    "Symmetry" : "Singlet",
    "NrOfExcitations" : 5,
    "StateOfInterest" : 1,
    "WriteSPTransitions" : "Yes",
    "WriteXplusY" : "Yes",
    "WriteXplusYAscii" : "Yes",
    "StateCouplings" : "{0 2}",
    
    "WriteAutotestTag" : "Yes",
    "WriteHS" : "Yes",
    "WriteEigenvectors" : "Yes",
    "EigenvectorsAsText" : "Yes",
    "PrintForces" : "Yes"
}
DFTB_methods.make_dftb_input(dftb_params)

## 3. Needed Slater-Koster files and system-specific parameters
<a name="3"></a>[Back to TOC](#toc)

We need to use specially-generated SK files. Here, we have two sets provided by Thomas Niehaus:

- mio-0-1_xSK.tgz
- ob2-1-1_xSK.tgz

In [6]:
#!tar -xf mio-0-1_xSK.tgz

Let's also define the path to the DFTB+ code. Change it, depending on the file system where you are running this tutorial

In [7]:
DFTB_EXE = "/home/alexvakimov/SOFTWARE/dftbplus/_install/bin/dftb+"
ODIN_EXE = "/home/alexvakimov/SOFTWARE/odin/odin"

## 4. Run DFTB+ calculations 
<a name="4"></a>[Back to TOC](#toc)

### 4.1. To generate H and S
<a name="4.1"></a>[Back to TOC](#toc)

<a name="run_dftb-1"></a>

In [8]:
help(DFTB_methods.run_dftb)

Help on function run_dftb in module libra_py.packages.dftbplus.methods:

run_dftb(coords, params)
    Execute a DFTB+ calculation in a thread-safe manner within a specified working directory.
    
    This function prepares input files, runs ground-state and excited-state
    DFTB+ calculations, and stores all outputs in a dedicated working directory.
    It is designed for parallel workflows (e.g., multi-trajectory simulations),
    where each trajectory runs independently without interfering with others.
    
    The function performs the following steps:
    
        1. Ensures the working directory exists.
        2. Writes a GEN-format geometry file using the provided coordinates.
        3. Generates a ground-state DFTB+ input file and executes DFTB+.
        4. Renames the ground-state output file ("autotest.tag") to
           "autotest_ground_state.tag".
        5. Generates an excited-state (TD-DFTB) input file and executes DFTB+.
        6. Captures standard output and error

We need to use the option `"what_calculation" : "gs"` in this kind of calculations, because the calculations are stopped after the ground state stage anyways.

**NOTE: The error message after this cell is normal as long as the expected files were generated**

In [9]:
atom_labels = ["O", "H", "H"]

coordinates = np.array([
    0.000000, 0.000000, 0.000000,
    0.758602, 0.000000, 0.504284,
   -0.758602, 0.000000, 0.504284
]).reshape(-1, 1) * units.Angst 

prms1 = { "labels" : atom_labels,
          "exe": DFTB_EXE,
          "dftb_run_params" : dftb_params,
          "working_directory" : "calc1",
          "what_calculation" : "gs"
        }
DFTB_methods.run_dftb(coordinates, prms1)

DFTB+ GEN file written to calc1/x1.gen


CalledProcessError: Command '['/home/alexvakimov/SOFTWARE/dftbplus/_install/bin/dftb+']' returned non-zero exit status 1.

### 4.2. To do SCF and TD-DFT calculations
<a name="4.2"></a>[Back to TOC](#toc)

In [10]:
dftb_params.update({"WriteHS" : "No"})
prms2 = { "labels" : atom_labels,
          "exe": DFTB_EXE,
          "dftb_run_params" : dftb_params,
          "working_directory" : "calc2",
        }
DFTB_methods.run_dftb(coordinates, prms2)

DFTB+ GEN file written to calc2/x1.gen


### 4.3. The overlap for the doubled-molecule - ODIN
<a name="4.3"></a>[Back to TOC](#toc)

In [11]:
double_labels = atom_labels*2
double_coords = np.concatenate( (coordinates, coordinates), axis=0)

print(double_labels)
print(double_coords)

['O', 'H', 'H', 'O', 'H', 'H']
[[ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 1.43354991]
 [ 0.        ]
 [ 0.95295858]
 [-1.43354991]
 [ 0.        ]
 [ 0.95295858]
 [ 0.        ]
 [ 0.        ]
 [ 0.        ]
 [ 1.43354991]
 [ 0.        ]
 [ 0.95295858]
 [-1.43354991]
 [ 0.        ]
 [ 0.95295858]]


Next, we will use two auxiliary functions: `create_odin_inp` 
<a name="create_odin_inp-1"></a>

In [12]:
help(DFTB_methods.create_odin_inp)

Help on function create_odin_inp in module libra_py.packages.dftbplus.methods:

create_odin_inp(params)
    Generate the input string for the ODIN overlap program in a thread-safe manner.
    
    This function constructs the text input required by the ODIN code
    (https://github.com/thomas-niehaus/odin) for computing atomic orbital
    overlap matrices from a GEN-format geometry file and Slater–Koster data.
    
    The function is designed to be safe for parallel execution by avoiding
    any dependence on the global working directory. It explicitly accesses
    files using a provided working directory while keeping file paths
    relative within the ODIN input itself.
    
    The function performs the following steps:
    
        1. Resolves the full path to the GEN geometry file using the provided
           working directory.
        2. Reads the GEN file and extracts the list of atomic element symbols
           from the second line.
        3. Maps each element to its maximu

and `run_odin`
<a name="run_odin-1"></a>

In [13]:
help(DFTB_methods.run_odin)

Help on function run_odin in module libra_py.packages.dftbplus.methods:

run_odin(params)
    Execute the ODIN program in a thread-safe manner within a specified working directory.
    
    This function prepares the ODIN input file, executes the ODIN binary,
    and produces atomic orbital overlap data (e.g., "oversqr.dat") required
    for nonadiabatic dynamics and electronic structure post-processing.
    
    The function is designed for parallel workflows (e.g., multi-trajectory
    simulations) by avoiding any dependence on the global working directory.
    All file operations and execution occur inside a user-specified directory.
    
    The function performs the following steps:
    
        1. Ensures the working directory exists (creates it if necessary).
        2. Generates the ODIN input string using ``create_odin_inp``.
        3. Writes the input to "odin.inp" inside the working directory.
        4. Executes the ODIN program using the input file via standard input.
   

In [14]:
wd = "calc3"

# Create working directory, if doesn't exist
if not os.path.exists(wd):
    os.mkdir(wd)

# Go into that directory
#os.chdir(wd)

# Make double geometry gen file
DFTB_methods.write_dftb_gen(F"{wd}/doubled_geometry.gen", double_labels, double_coords/units.Angst)

# Make input for Odin:
odin_params = {
    "ODIN_EXE": ODIN_EXE,
    "working_directory" : wd, 
    "filename":"doubled_geometry.gen",
    "slakos_prefix" : "../mio/FinalSK/",
    "max_ang_mom": {"H":1, "O":2},
}

## This line may be commented since the ODIN input file would be created 
## by the `run_odin` function
DFTB_methods.create_odin_inp(odin_params)

# Run Odin
DFTB_methods.run_odin(odin_params)

# Go back to the original directory
#os.chdir("../")

DFTB+ GEN file written to calc3/doubled_geometry.gen


### 4.4. Check time-overlap
<a name="4.4"></a>[Back to TOC](#toc)

First, let's use one of the auxiliary functions to get the dimension of the overlap matrix (the number of AOs):
<a name="read_spx_mappings-1"></a>

In [15]:
rpa_map, _, _ = DFTB_methods.read_spx_mappings("calc2/SPX.DAT")
ndim = rpa_map.shape[0]
print(ndim)

6


Then, we can read the matrix. Note that since this is the overlap matrix for the doubled molecule, we use doubled dimension `2*ndim`.

Once it is read, we read sub-blocks of the bigger overlap matrix. The diagonal blocks are the self-overlaps (of the same geometry with itself) just for different geometries (first and second). 

The off-diagonal blocks are the overlaps of AOs between two geometries first and second.

Since in our input above the firs and second geometries are identical, we expect to have all the block matrices to be alike.
<a name="read_overlap_matrix-1"></a>

In [16]:
## Overlap matrix
S = DFTB_methods.read_overlap_matrix('calc3/oversqr.dat', 2*ndim)
print(S.shape)
print(S)
print("S00")
print(S[:ndim, :ndim])
print("S01")
print(S[:ndim, ndim:2*ndim])
print("S10")
print(S[ndim:2*ndim, :ndim])
print("S11")
print(S[ndim:2*ndim, ndim:2*ndim])

(12, 12)
[[ 1.          0.          0.          0.          0.46501291  0.46501291
   1.          0.          0.          0.          0.46501291  0.46501291]
 [ 0.          1.          0.          0.          0.          0.
   0.          1.          0.          0.          0.          0.        ]
 [ 0.          0.          1.          0.          0.22860337  0.22860337
   0.          0.          1.          0.          0.22860337  0.22860337]
 [ 0.          0.          0.          1.          0.34389149 -0.34389149
   0.          0.          0.          1.          0.34389149 -0.34389149]
 [ 0.46501291  0.          0.22860337  0.34389149  1.          0.19317855
   0.46501291  0.          0.22860337  0.34389149  1.          0.19317855]
 [ 0.46501291  0.          0.22860337 -0.34389149  0.19317855  1.
   0.46501291  0.          0.22860337 -0.34389149  0.19317855  1.        ]
 [ 1.          0.          0.          0.          0.46501291  0.46501291
   1.          0.          0.          

### 4.5. Extracting MO/CI data in the required format
<a name="4.5"></a>[Back to TOC](#toc)

We use the `DFTB_methods.read_dftb_orbital_info` function to read the results of the TD-DFTB calculations in the format expected in the workflow:
<a name="read_dftb_orbital_info-1"></a>

In [17]:
help(DFTB_methods.read_dftb_orbital_info)

Help on function read_dftb_orbital_info in module libra_py.packages.dftbplus.methods:

read_dftb_orbital_info(params_)
    Read molecular orbital (MO) information, configurations, and CI amplitudes
    from DFTB+ excited-state output files.
    
    This function extracts the active-space MO coefficients and excited-state
    configuration interaction (CI) information from DFTB+ linear-response
    (Casida/RPA) output files.
    
    Parameters
    ----------
    params_ : dict
        Dictionary of input parameters. Recognized keys:
    
        source_directory : str, optional, default="calc"
            Directory containing the DFTB+ output files:
                - SPX.DAT      : orbital excitation mappings
                - eigenvec.bin : MO coefficient matrix
                - XplusY.DAT   : excitation energies and CI vectors
    
        orbital_space : list of int, optional, default=None
            List of molecular orbital indices (1-based indexing) defining
            the ac

In [18]:
info, mo1, data1 = DFTB_methods.read_dftb_orbital_info({"nstates":5, "orbital_space":None, "source_directory":"calc2" })

In [19]:
print(mo1)
print(data1[0])
print(data1[1])
print(data1[2])
print(info)

[[ 8.33685274e-01 -2.77555756e-17  2.69865325e-01 -7.65648513e-17
  -2.20994626e-16 -9.44042104e-01]
 [ 0.00000000e+00  1.66533454e-16 -4.16333634e-16 -1.00000000e+00
   5.55111512e-17  0.00000000e+00]
 [-1.14774356e-03 -7.17362068e-18 -8.94142966e-01  4.04328166e-16
  -1.46535640e-16 -5.99824914e-01]
 [ 3.55467454e-18 -6.43528029e-01  5.48717889e-17 -2.80384010e-16
  -1.00030346e+00  3.66072163e-16]
 [ 1.58238665e-01 -3.87758479e-01 -1.74442914e-01  3.49079012e-17
   8.52277813e-01  8.40541619e-01]
 [ 1.58238665e-01  3.87758479e-01 -1.74442914e-01  4.52091732e-17
  -8.52277813e-01  8.40541619e-01]]
[0.83071084 0.88813661 1.0445667  1.04645814]
[array([[4, 5],
       [3, 5],
       [2, 5],
       [4, 6],
       [3, 6],
       [2, 6],
       [1, 5],
       [1, 6]]), array([[4, 5],
       [3, 5],
       [2, 5],
       [4, 6],
       [3, 6],
       [2, 6],
       [1, 5],
       [1, 6]]), array([[4, 5],
       [3, 5],
       [2, 5],
       [4, 6],
       [3, 6],
       [2, 6],
       [1, 5

In [20]:
data1[0]

array([0.83071084, 0.88813661, 1.0445667 , 1.04645814])

However, in realistic calculations the number of configurations may be very large, so the calculations will be more expensive for the two reasons:

- the bare number of configurations will increase (so the size of the CI amplitude matrices will increase)

- more critically: since the excitations from "deeper" levels will be included, the size of the Slater determinant will be larger - more electrons will need to be included explicitly.

Reading with CI threshold:

In [21]:
info, mo1, data1 = DFTB_methods.read_dftb_orbital_info({"nstates":5, "orbital_space":None, 
                                                        "source_directory":"calc2",
                                                        "ci_threshold":0.01})
print(data1[1])
print(data1[2])
print(info)

[array([[4, 5]]), array([[3, 5],
       [2, 6],
       [1, 5]]), array([[2, 5],
       [3, 6],
       [1, 6]]), array([[4, 6]])]
[array([-1.]), array([-0.99267763,  0.0660317 , -0.03828367]), array([-0.89191693,  0.41936233, -0.07312469]), array([-1.])]
{'nocc': np.int64(4), 'nelec': np.int64(8), 'nao': 6, 'nmo': 6, 'nci': 4, 'nact': 6, 'actual_orbital_space': [1, 2, 3, 4, 5, 6], 'min_occ': np.int64(1), 'max_occ': np.int64(4), 'min_vir': np.int64(5), 'max_vir': np.int64(6)}


## 5. Putting everything together
<a name="5"></a>[Back to TOC](#toc)

### 5.1. Develop the function
<a name="5.1"></a>[Back to TOC](#toc)

In [22]:
class tmp:
    pass


def dftb_compute_adi(q, params, full_id):
    """
    Compute adiabatic electronic properties and derivative couplings for one trajectory
    using DFTB+ electronic structure and ODIN orbital overlap calculations.

    This function performs a single-time-step electronic structure evaluation
    for a trajectory in nonadiabatic dynamics. It computes molecular orbitals (MOs),
    CI states, and their overlaps between consecutive time steps, constructing
    the adiabatic Hamiltonian, vibronic Hamiltonian, and derivative couplings.

    The function is designed for trajectory-based nonadiabatic methods, such as:
        - FSSH (Fewest Switches Surface Hopping)
        - Ehrenfest dynamics
        - Mapping-based methods
        - Exact factorization / quantum trajectory approaches

    Workflow
    --------
    1. Extract nuclear coordinates for the trajectory.
    2. Run ground-state and excited-state DFTB+ calculations in a trajectory-specific directory.
    3. Generate a "doubled" geometry file to compute AO overlaps with ODIN.
    4. Read molecular orbitals, CI coefficients, and excitation energies.
    5. Compute:
        - Time-overlap matrices between consecutive CI states
        - Adiabatic Hamiltonian
        - Vibronic Hamiltonian
        - Forces from DFTB+ outputs
        - Derivative couplings (from NACVs and time-overlap)
    6. Update trajectory-specific previous-state data in `params`.

    Parameters
    ----------
    q : MATRIX
        Nuclear coordinates for all trajectories.
        Shape: (3*N_atoms, N_trajectories)
        Units: Bohr
        Column `itraj` corresponds to trajectory `itraj`.

    params : dict
        Dictionary of simulation parameters and trajectory state information.
        Keys used include:

        Required:
        ---------
        atom_labels : list of str
            Atomic symbols, e.g., ["O", "H", "H"].
        orbital_space : dict
            Active-space molecular orbitals indices.

        Optional / Internal (updated in-place):
        --------------------------------------
        dt : float, default=41.0
            Nuclear time step in atomic units.
        dftb_run_params : dict
            Parameters for DFTB+ calculations, e.g. NrOfExcitations, sk_prefix.
        dftb_exe : str, default="dftb+"
            Path to DFTB+ executable.
        odin_exe : str, default="odin"
            Path to ODIN executable.
        working_directory_prefix : str, default="wd"
            Prefix for trajectory-specific directories.
        is_first_time : dict
            Dictionary keyed by trajectory index (`itraj`) with boolean values.
            True indicates that the current step is the first step of this trajectory.
        act_state : dict
            Dictionary keyed by trajectory index (`itraj`) with integer values
            indicating the active electronic state for this trajectory.
        MO_prev : dict
            Previous molecular orbitals per trajectory (updated in-place).
        data_prev : dict
            Previous CI data per trajectory (updated in-place).
        coordinates_prev : dict
            Previous nuclear coordinates per trajectory (updated in-place).
        ci_threshold : float, default=0.01
            Threshold for CI truncation.
        odin_max_ang_mom : dict, optional
            Maximum angular momentum per element for ODIN.

    full_id : int or object
        Encoded trajectory identifier (decoded to extract `itraj`).

    Returns
    -------
    obj : tmp
        Object containing adiabatic electronic properties for this trajectory.
        Attributes include:

        ham_adi : CMATRIX (nstates, nstates)
            Adiabatic Hamiltonian matrix.

        hvib_adi : CMATRIX (nstates, nstates)
            Vibronic Hamiltonian including derivative couplings.

        time_overlap_adi : CMATRIX (nstates, nstates)
            Time-overlap matrix S_ij(t, t+dt) = <Ψ_i(t)|Ψ_j(t+dt)>.

        basis_transform : CMATRIX (nstates, nstates)
            Basis transformation matrix (currently identity).

        d1ham_adi : CMATRIXList
            List of derivative Hamiltonians with respect to nuclear coordinates.

        dc1_adi : CMATRIXList
            List of derivative couplings for each nuclear degree of freedom.

    Notes
    -----
    - All computations are performed in trajectory-specific directories to
      ensure thread safety.
    - Ground-state (state 0) and excited-state (states 1..nstates-1) calculations
      are run sequentially using DFTB+.
    - Overlap matrices are computed by doubling the geometry (previous + current step)
      and running ODIN.
    - Forces are extracted from `autotest.tag` for the active state.
    - Derivative couplings are estimated from the anti-symmetric part of the
      time-overlap matrix divided by 2*dt:
          Hvib_ij = E_i δ_ij - i d_ij
          d_ij = (S_ij - S_ji) / (2 dt)
    - Energies are in Hartree, time in atomic units, coordinates in Bohr,
      and overlaps are dimensionless.
    - `is_first_time` and `act_state` are dictionaries keyed by trajectory index,
      allowing simultaneous tracking of multiple trajectories in parallel computations.

    Example
    -------
    >>> obj = dftb_compute_adi_new(q, params, full_id)
    >>> print(obj.ham_adi)
    >>> print(obj.dc1_adi[0])
    """
    # ================= Decode trajectory index =================
    Id = Cpp2Py(full_id)
    itraj = Id[-1]

    # ================= Extract coordinates =================
    coords = q.col(itraj)
    coordinates = data_conv.MATRIX2nparray(coords, float)

    ndof = coords.num_of_rows
    nat = ndof // 3

    # ================= Safe param access =================
    params.setdefault("MO_prev", {})
    params.setdefault("data_prev", {})
    params.setdefault("coordinates_prev", {})
    params.setdefault("s_ci_inv_prev", {})
    params.setdefault("is_first_time", {})
    params.setdefault("act_state", {})

    is_first_time = params["is_first_time"].get(itraj, True)
    act_state = params["act_state"].get(itraj, 0)

    # ================= Read parameters =================
    dt = float(params.get("dt", 41.0))
    dftb_run_params = params.get("dftb_run_params", {})
    atom_labels = params["atom_labels"]

    wd_prefix = params.get("working_directory_prefix", "wd")
    wd = f"{wd_prefix}_itraj{itraj}"

    # ================= Run DFTB+ =================
    dftb_params = copy.deepcopy(dftb_run_params)
    dftb_params["StateOfInterest"] = act_state

    #DFTB_methods.make_dftb_input(dftb_params)
    prms1 = {
        "labels": atom_labels,
        "exe": params.get("dftb_exe", "dftb+"),
        "dftb_run_params": dftb_params,
        "working_directory": wd,
        "gen_file": dftb_run_params.get("gen_file", "x1.gen"),
    }
    DFTB_methods.run_dftb(coordinates, prms1)

    # ================= Previous coordinates =================
    if is_first_time:
        coordinates_prev = coordinates.copy()
    else:
        coordinates_prev = params["coordinates_prev"].get(itraj, coordinates).copy()

    # ================= ODIN =================
    double_labels = atom_labels * 2
    double_coords = np.concatenate((coordinates_prev, coordinates), axis=0)

    gen_path = os.path.join(wd, "doubled_geometry.gen")

    DFTB_methods.write_dftb_gen(
        gen_path,
        double_labels,
        double_coords / units.Angst
    )
    
    odin_params = {
        "ODIN_EXE": params.get("odin_exe", "odin"),
        "filename": "doubled_geometry.gen",
        "working_directory": wd,
        "slakos_prefix": dftb_run_params.get("sk_prefix", "../mio/FinalSK/"),
        "max_ang_mom": params.get("odin_max_ang_mom", {"H": 1, "O": 2}),
    }

    #DFTB_methods.create_odin_inp(odin_params)
    DFTB_methods.run_odin(odin_params)

    # ================= Read electronic structure =================
    nstates = dftb_run_params.get("NrOfExcitations", 1) + 1

    read_params = {
        "nstates": nstates,
        "orbital_space": params["orbital_space"],
        "source_directory": wd,
        "ci_threshold": params.get("ci_threshold", 0.01),
    }

    info, MO_curr, data_curr = DFTB_methods.read_dftb_orbital_info(read_params)

    # ================= Overlap =================
    ndim = info["nmo"]
    S = DFTB_methods.read_overlap_matrix(f"{wd}/oversqr.dat", 2 * ndim)

    st_ao = S[:ndim, ndim:]
    s_ao = S[ndim:, ndim:]

    # ================= Previous electronic data =================
    if is_first_time:
        MO_prev = MO_curr.copy()
        data_prev = copy.deepcopy(data_curr)
    else:
        MO_prev = params["MO_prev"].get(itraj, MO_curr).copy()
        data_prev = params["data_prev"].get(itraj, data_curr)

    # ================= Build object =================
    obj = tmp()

    obj.ham_adi = CMATRIX(nstates, nstates)
    obj.nac_adi = CMATRIX(nstates, nstates)
    obj.hvib_adi = CMATRIX(nstates, nstates)
    obj.basis_transform = CMATRIX(nstates, nstates)
    obj.time_overlap_adi = CMATRIX(nstates, nstates)
    obj.overlap_adi = CMATRIX(nstates, nstates)

    # ================= Compute overlaps =================
    s_mo_orb = MO_curr.T @ s_ao @ MO_curr
    s_mo = np.kron(np.eye(2), s_mo_orb)
    
    st_mo_orb = MO_prev.T @ st_ao @ MO_curr
    st_mo = np.kron(np.eye(2), st_mo_orb)
    
    print(scipy.linalg.det(st_mo_orb))
    print(st_mo_orb)
    print( np.max( np.abs(st_mo_orb[:,:]), axis=0) )

    ovlp_params = {
        "homo_indx": info["nocc"],
        "nocc": info["nocc"] - 1,
        "nvirt": info["nmo"] - info["nocc"],
        "nelec": info["nelec"],
        "nstates": nstates,
        "active_space": info["actual_orbital_space"],
    }

    st_ci = ci.overlap(st_mo, data_prev, data_curr, ovlp_params)
    s_ci = ci.overlap(s_mo, data_curr, data_curr, ovlp_params)
    
    s_ci_inv_curr = ortho.lowdin_inverse_sqrt(s_ci)
    s_ci_inv_prev = None
    if is_first_time:
        s_ci_inv_prev = copy.deepcopy(s_ci_inv_curr)
    else:
        s_ci_inv_prev = params["s_ci_inv_prev"].get(itraj, s_ci_inv_curr)

    s_ci = s_ci_inv_curr @ s_ci @ s_ci_inv_curr
    st_ci = s_ci_inv_prev @ st_ci @ s_ci_inv_curr
    

    # ================= Populate Hamiltonian =================
    for i in range(nstates):
        energy = 0.0 if i == 0 else 0.5 * (
            data_prev[0][i - 1] + data_curr[0][i - 1]
        )

        obj.ham_adi.set(i, i, energy)
        obj.hvib_adi.set(i, i, energy)
        obj.basis_transform.set(i, i, 1.0)

        for j in range(nstates):
            obj.time_overlap_adi.set(i, j, float(st_ci[i, j]))
            obj.overlap_adi.set(i,  j, float(s_ci[i,j]) )

    # ================== Forces ===============================
    ## autotest.tag
    results_gs, results_es = None, None
    results_gs = DFTB_methods.parse_tagged_file(f"{wd}/autotest_ground_state.tag")
    results_ex = DFTB_methods.parse_tagged_file(f"{wd}/autotest.tag")

    #print(F"results_gs = {results_gs}")
    #print(F"results_ex = {results_ex}")

    forces = None
    if act_state == 0:
        forces = results_gs['forces']
    else:
        forces = results_ex['forces']

    e0 = results_gs['mermin_energy']
    print(F"GS energy = {e0}")
    for i in range(nstates):
        obj.ham_adi.add(i, i, e0)


    obj.d1ham_adi = CMATRIXList()
    for idof in range(ndof):
        obj.d1ham_adi.append(CMATRIX(nstates, nstates))

    if os.path.exists(f"{wd}/FRC.DAT"):
        forces = DFTB_methods.read_all_forces(f"{wd}/FRC.DAT")
        for iatom in range(nat):
            for i in range(nstates):
                obj.d1ham_adi[3 * iatom + 0].set(i, i, -forces[i, 0, iatom] * (1.0 + 0.0j))
                obj.d1ham_adi[3 * iatom + 1].set(i, i, -forces[i, 1, iatom] * (1.0 + 0.0j))
                obj.d1ham_adi[3 * iatom + 2].set(i, i, -forces[i, 2, iatom] * (1.0 + 0.0j))
                
    # ================= Derivative couplings ====================
    obj.dc1_adi = CMATRIXList()
    for idof in range(ndof):
        obj.dc1_adi.append(CMATRIX(nstates, nstates))

    if os.path.exists(f"{wd}/NACV.DAT"):
        nacv = DFTB_methods.read_nacv(f"{wd}/NACV.DAT")
        for iatom in range(nat):
            for i in range(nstates):
                for j in range(nstates):
                    obj.dc1_adi[3 * iatom + 0].set(i, j, nacv[i,j,0,iatom] * (1.0 + 0.0j))
                    obj.dc1_adi[3 * iatom + 1].set(i, j, nacv[i,j,1,iatom] * (1.0 + 0.0j))
                    obj.dc1_adi[3 * iatom + 2].set(i, j, nacv[i,j,2,iatom] * (1.0 + 0.0j))

    # ================= Compute derivative couplings =================
    for i in range(nstates):
        for j in range(i + 1, nstates):
            dij = ( obj.time_overlap_adi.get(i, j) - obj.time_overlap_adi.get(j, i) ) / (2.0 * dt)
            obj.hvib_adi.set(i, j, -1j * dij)
            obj.hvib_adi.set(j, i, +1j * dij)            
            
    # ================= Store state =================
    params["MO_prev"][itraj] = MO_curr.copy()
    params["data_prev"][itraj] = copy.deepcopy(data_curr)
    params["coordinates_prev"][itraj] = coordinates.copy()
    params["s_ci_inv_prev"][itraj] = copy.deepcopy(s_ci_inv_curr)
    params["is_first_time"][itraj] = False

    return obj


### 5.2. Test the funciton
<a name="5.2"></a>[Back to TOC](#toc)

In [23]:
os.chdir("/home/alexvakimov/CCCT/Tutorials_Libra/11_program_specific_methods/4_dftbplus_methods/3_workflow")
labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", 0)

print(labels)
print(q)

dftb_params = {
    "gen_file" : "x1.gen",
    "sk_prefix" : "../mio/FinalSK/",
    "Driver" : "{}",
    "MaxAngularMomentum" : """{ C = "p"
                                H = "s"
                     }
                     """,
    "Symmetry" : "Singlet",
    "NrOfExcitations" : 5,
    "StateOfInterest" : 1,
    "WriteSPTransitions" : "Yes",
    "WriteXplusY" : "Yes",
    "WriteXplusYAscii" : "Yes",
    "StateCouplings" : "{0 5}",
    
    "WriteAutotestTag" : "Yes",
    "WriteHS" : "No",
    "WriteEigenvectors" : "Yes",
    "EigenvectorsAsText" : "Yes",
    "PrintForces" : "Yes",
    "Filling" : """Fermi { Temperature [K] = 0.5 }
                """
}

params = {"atom_labels":labels, "timestep":0, 
               "dftb_exe":"/home/alexvakimov/SOFTWARE/dftbplus/_install/bin/dftb+", 
               "dftb_run_params": dftb_params,
               "working_directory":"dftb_workflow",
               
               "odin_exe": "/home/alexvakimov/SOFTWARE/odin/odin",
               "odin_max_ang_mom" : { "C":2, "H":1 },
               "orbital_space" : None,
               
               "dt":1.0*units.fs2au,
               "nelec_act_space":None,
               "ci_threshold":0.01,
               
               "is_first_time":{0:True},
               "act_state":{0:1},
               "read_forces" : False,
               "read_nacvs": False,
              }
print(params)

# Emulates 1 trajectory
full_id = Py2Cpp_int([0, 0])

res = "results"
# Create working directory, if doesn't exist
if not os.path.exists(res):
    os.mkdir(res)

# Do the first 5 steps 
for i in range(5):
    print(F"======== Iteration {i} ==============")
    labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", i)
    params["timestep"] = i
    
    obj = dftb_compute_adi(q, params, full_id)        
    obj.ham_adi.show_matrix(F"{res}/ham_adi_{i}.txt")
    obj.hvib_adi.show_matrix(F"{res}/hvib_adi_{i}.txt")
    obj.time_overlap_adi.real().show_matrix(F"{res}/st_adi_{i}.txt")
    obj.overlap_adi.real().show_matrix(F"{res}/s_adi_{i}.txt")
    

['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
{'atom_labels': ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H'], 'timestep': 0, 'dftb_exe': '/home/alexvakimov/SOFTWARE/dftbplus/_install/bin/dftb+', 'dftb_run_params': {'gen_file': 'x1.gen', 'sk_prefix': '../mio/FinalSK/', 'Driver': '{}', 'MaxAngularMomentum': '{ C = "p"\n                                H = "s"\n                     }\n                     ', 'Symmetry': 'Singlet', 'NrOfExcitations': 5, 'StateOfInterest': 1, 'WriteSPTransitions': 'Yes', 'WriteXplusY': 'Yes', 'WriteXplusYAscii': 'Yes', 'StateCouplings': '{0 5}', 'WriteAutotestTag': 'Yes', 'WriteHS': 'No', 'WriteEigenvectors': 'Yes', 'EigenvectorsAsText': 'Yes', 'PrintForces': 'Yes', 'Filling': 'Fermi { Temperature [K] = 0.5 }\n                '}, 'working_directory': 'dftb_workflow', 'odin_exe': '/home/alexva

/tmp/ipykernel_234309/2062045261.py:289: ComplexWarning: Casting complex values to real discards the imaginary part
  obj.time_overlap_adi.set(i, j, float(st_ci[i, j]))
/tmp/ipykernel_234309/2062045261.py:290: ComplexWarning: Casting complex values to real discards the imaginary part
  obj.overlap_adi.set(i,  j, float(s_ci[i,j]) )


GS energy = -23.2766241727941
======== Iteration 1 ==============
DFTB+ GEN file written to wd_itraj0/x1.gen
DFTB+ GEN file written to wd_itraj0/doubled_geometry.gen
0.9817203312791429
[[ 9.99905605e-01  1.28379702e-03 -2.45713978e-03 ...  2.68752794e-04
   2.97227133e-05  3.98048300e-04]
 [-5.75936393e-04 -2.59174307e-01 -1.43792087e-01 ... -3.43190088e-04
  -6.87107180e-04  4.64272544e-04]
 [ 1.83648370e-03 -9.58499638e-01  1.58867974e-01 ... -6.93373456e-04
   6.75094411e-04 -2.34163658e-04]
 ...
 [ 2.97699769e-04  8.15844203e-04  4.13023377e-04 ... -8.02142896e-01
   4.25449722e-01 -3.32022762e-02]
 [ 2.45831675e-05 -6.84204790e-04  3.99298432e-04 ...  1.41844505e-01
  -4.19127738e-01  3.83651447e-02]
 [-3.85213926e-04 -1.21964902e-04 -2.68954213e-04 ... -2.13649420e-02
   7.67600548e-02  9.88702658e-01]]
[0.9999056  0.95849964 0.97662404 0.95496387 0.9912876  0.99138722
 0.74611287 0.81658872 0.85695403 0.99948341 0.99622648 0.8295355
 0.82992906 0.85313139 0.85302881 0.99563887 0

### 5.3. Test Libra implementation
<a name="5.3"></a>[Back to TOC](#toc)
<a name="dftb_compute_adi-1"></a>

In [24]:
os.chdir("/home/alexvakimov/CCCT/Tutorials_Libra/11_program_specific_methods/4_dftbplus_methods/3_workflow")
labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", 0)

print(labels)
print(q)

dftb_params = {
    "gen_file" : "x1.gen",
    "sk_prefix" : "../mio/FinalSK/",
    "Driver" : "{}",
    "MaxAngularMomentum" : """{ C = "p"
                                H = "s"
                     }
                     """,
    "Symmetry" : "Singlet",
    "NrOfExcitations" : 5,
    "StateOfInterest" : 1,
    "WriteSPTransitions" : "Yes",
    "WriteXplusY" : "Yes",
    "WriteXplusYAscii" : "Yes",
    "StateCouplings" : "{0 5}",
    
    "WriteAutotestTag" : "Yes",
    "WriteHS" : "No",
    "WriteEigenvectors" : "Yes",
    "EigenvectorsAsText" : "Yes",
    "PrintForces" : "Yes",
    "Filling" : """Fermi { Temperature [K] = 0.5 }
                """
}

params = {"atom_labels":labels, "timestep":0, 
               "dftb_exe":"/home/alexvakimov/SOFTWARE/dftbplus/_install/bin/dftb+", 
               "dftb_run_params": dftb_params,
               "working_directory":"dftb_workflow",
               
               "odin_exe": "/home/alexvakimov/SOFTWARE/odin/odin",
               "odin_max_ang_mom" : { "C":2, "H":1 },
               "orbital_space" : None,
               
               "dt":1.0*units.fs2au,
               "nelec_act_space":None,
               "ci_threshold":0.01,
               
               "is_first_time":{0:True},
               "act_state":{0:1},
               "read_forces" : False,
               "read_nacvs": False,
              }
print(params)

# Emulates 1 trajectory
full_id = Py2Cpp_int([0, 0])

res = "results2"
# Create working directory, if doesn't exist
if not os.path.exists(res):
    os.mkdir(res)

# Do the first 5 steps 
for i in range(5):
    print(F"======== Iteration {i} ==============")
    labels, q = cp2k.read_trajectory_xyz_file("adamantane.xyz", i)
    params["timestep"] = i
    
    obj = DFTB_methods.dftb_compute_adi(q, params, full_id)        
    obj.ham_adi.show_matrix(F"{res}/ham_adi_{i}.txt")
    obj.hvib_adi.show_matrix(F"{res}/hvib_adi_{i}.txt")
    obj.time_overlap_adi.real().show_matrix(F"{res}/st_adi_{i}.txt")
    obj.overlap_adi.real().show_matrix(F"{res}/s_adi_{i}.txt")
    

['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']
{'atom_labels': ['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H'], 'timestep': 0, 'dftb_exe': '/home/alexvakimov/SOFTWARE/dftbplus/_install/bin/dftb+', 'dftb_run_params': {'gen_file': 'x1.gen', 'sk_prefix': '../mio/FinalSK/', 'Driver': '{}', 'MaxAngularMomentum': '{ C = "p"\n                                H = "s"\n                     }\n                     ', 'Symmetry': 'Singlet', 'NrOfExcitations': 5, 'StateOfInterest': 1, 'WriteSPTransitions': 'Yes', 'WriteXplusY': 'Yes', 'WriteXplusYAscii': 'Yes', 'StateCouplings': '{0 5}', 'WriteAutotestTag': 'Yes', 'WriteHS': 'No', 'WriteEigenvectors': 'Yes', 'EigenvectorsAsText': 'Yes', 'PrintForces': 'Yes', 'Filling': 'Fermi { Temperature [K] = 0.5 }\n                '}, 'working_directory': 'dftb_workflow', 'odin_exe': '/home/alexva

/home/alexvakimov/SOFTWARE/libra/_build/src/libra_py/packages/dftbplus/methods.py:2963: ComplexWarning: Casting complex values to real discards the imaginary part
  obj.time_overlap_adi.set(i, j, float(st_ci[i, j]))
/home/alexvakimov/SOFTWARE/libra/_build/src/libra_py/packages/dftbplus/methods.py:2964: ComplexWarning: Casting complex values to real discards the imaginary part
  obj.overlap_adi.set(i,  j, float(s_ci[i,j]) )


GS energy = -23.2766241727941
======== Iteration 1 ==============
DFTB+ GEN file written to wd_itraj0/x1.gen
DFTB+ GEN file written to wd_itraj0/doubled_geometry.gen
GS energy = -23.274020118486
======== Iteration 2 ==============
DFTB+ GEN file written to wd_itraj0/x1.gen
DFTB+ GEN file written to wd_itraj0/doubled_geometry.gen
GS energy = -23.2684813429252
======== Iteration 3 ==============
DFTB+ GEN file written to wd_itraj0/x1.gen
DFTB+ GEN file written to wd_itraj0/doubled_geometry.gen
GS energy = -23.2635073855038
======== Iteration 4 ==============
DFTB+ GEN file written to wd_itraj0/x1.gen
DFTB+ GEN file written to wd_itraj0/doubled_geometry.gen
GS energy = -23.260764259567
